<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/7%EC%A3%BC%EC%B0%A8_%EB%B3%B5%EC%8A%B5%EA%B3%BC%EC%A0%9C_Transformer%EC%9B%90%EB%A6%AC%EC%99%80ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7주차 복습 과제: Transformer의 원리와 ChatGPT 답변 원리 파악

> **복습 키워드**
> 1. Transformer의 원리와 개념을 통해 ChatGPT 모델이 어떻게 답변을 하는지에 대한 원리 파악
> 2. 딥러닝의 구조를 파악함으로써 향후 RAG 및 파인튜닝 등의 이해 도모


---
## 1. 왜 Transformer가 등장했는가?

텍스트를 처리하는 딥러닝 모델은 크게 세 단계로 발전해 왔습니다.

### 기존 방식의 한계

**BoW / TF-IDF 방식**은 단어의 빈도만 세기 때문에 단어 순서를 무시합니다.
"영화가 지루하지 않고 재미있다"(긍정)와 "영화가 재미있지 않고 지루하다"(부정)를 구별할 수 없습니다.

**RNN / LSTM 방식**은 단어를 순서대로 하나씩 읽으면서 문맥을 누적합니다.
순서를 반영할 수 있지만, 문장이 길어지면 앞쪽 단어의 정보가 점점 사라지는 **장기 의존성 문제**가 있습니다.
또한 단어를 하나씩 처리하므로 **병렬 처리가 불가능**하여 속도가 느립니다.

**Transformer**는 이 두 가지 문제를 모두 해결합니다.
모든 단어를 **동시에** 보면서 단어 간 관계를 직접 계산하기 때문에, 아무리 긴 문장이라도 먼 거리의 단어를 직접 참조할 수 있고, 병렬 처리로 학습 속도도 빠릅니다.

### 핵심 차이 비교

| 구분 | LSTM | Transformer |
|------|------|-------------|
| 읽는 방식 | 순서대로 하나씩 | 모든 단어를 동시에 |
| 긴 문장 처리 | 앞쪽 정보 유실 (기울기 소실) | 어떤 단어든 직접 참조 가능 |
| 병렬 처리 | 불가능 (느림) | 가능 (빠름) |
| 핵심 기술 | 게이트 메커니즘 (망각/입력/출력) | Self-Attention |
| 위치 정보 | 순서대로 읽어서 자동 인식 | Positional Encoding으로 별도 추가 |


---
## 2. Self-Attention — Transformer의 핵심 원리

### 2-1. Self-Attention이란?

Self-Attention은 **"이 단어를 이해하려면 문장 내 어떤 단어를 참고해야 할까?"**를 계산하는 메커니즘입니다.

예를 들어 "그 영화는 배우의 연기가 훌륭해서 재미있었다"라는 문장에서,
"재미있었다"라는 단어를 이해하기 위해 "연기가"와 "훌륭해서"에 더 많이 주목해야 합니다.
Self-Attention은 이런 단어 간 관련도를 자동으로 학습합니다.

### 2-2. Q, K, V의 역할

Self-Attention은 각 단어를 세 가지 역할로 변환해서 사용합니다.

| 역할 | 이름 | 비유 |
|------|------|------|
| **Q (Query)** | 질문 | "나는 누구와 관련 있지?" — 정보를 찾으려는 단어 |
| **K (Key)** | 키 | "나는 이런 특징을 가지고 있어" — 검색 대상의 라벨 |
| **V (Value)** | 값 | "내가 전달할 실제 정보" — 실제로 합쳐질 내용 |

각 단어의 임베딩 벡터에 학습 가능한 가중치 행렬(W_Q, W_K, W_V)을 곱해서 Q, K, V를 만듭니다.
같은 입력이지만 역할에 따라 다르게 변환되는 것이 핵심입니다.

### 2-3. 계산 과정 (수식)

```
Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```

이 수식은 네 단계로 이루어집니다.

**① Q와 K의 내적 (유사도 점수)**
각 단어 쌍의 관련도를 점수로 매깁니다. Q와 K의 내적값이 클수록 두 단어가 서로 관련이 높다는 의미입니다.

**② 스케일링 (sqrt(d_k)로 나누기)**
내적값이 너무 커지면 softmax 결과가 극단적(0 또는 1에 가까움)이 되어 학습이 어렵습니다.
벡터 차원의 제곱근으로 나눠서 값의 크기를 조절합니다.

**③ Softmax (확률로 변환)**
각 단어가 다른 단어에 주목하는 비율을 합이 1인 확률로 변환합니다.
예: "재미있다"가 "정말"에 0.6, "영화가"에 0.3, 자기 자신에 0.1 주목.

**④ 가중 합산 (확률 × V)**
주목 비율대로 V(실제 정보)를 합쳐서, 문맥이 반영된 새로운 벡터를 만듭니다.
결과적으로 각 단어는 자기 정보만이 아니라 관련 단어의 정보까지 포함하게 됩니다.


### 2-4. Self-Attention 직접 계산 실습

아래 코드를 실행하여 "영화가 정말 재미있다"라는 문장에서 Self-Attention이 실제로 어떻게 동작하는지 확인합니다.


In [ ]:
!pip install koreanize-matplotlib
import koreanize_matplotlib


In [ ]:
import numpy as np
np.random.seed(42)

# 각 단어를 4차원 벡터로 표현 (임베딩)
words = ["영화가", "정말", "재미있다"]
X = np.array([
    [1.0, 0.5, -0.3, 0.8],
    [0.2, -0.1, 0.9, 0.4],
    [0.7, 0.3, 0.6, -0.5],
])

d_k = 3

# 가중치 행렬로 Q, K, V 생성
W_Q = np.random.randn(4, d_k) * 0.5
W_K = np.random.randn(4, d_k) * 0.5
W_V = np.random.randn(4, d_k) * 0.5

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# 유사도 → 스케일링 → softmax
scores = Q @ K.T / np.sqrt(d_k)

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

weights = softmax(scores)

# 가중 합산
output = weights @ V

print("Attention 가중치 (각 단어가 다른 단어에 주목하는 비율):")
print(weights.round(3))
print()
for i, w in enumerate(words):
    top = words[np.argmax(weights[i])]
    print(f"  '{w}' → '{top}'에 가장 주목 ({weights[i].max():.0%})")
print()
print("→ Self-Attention 후, 각 단어는 관련 단어의 정보가 섞인 새로운 벡터를 갖게 됩니다.")


---
## 3. Multi-Head Attention — 여러 관점에서 동시에 보기

하나의 Attention만으로는 단어 간 관계를 한 가지 관점에서만 파악합니다.
예를 들어 "나는 어제 카페에서 맛있는 커피를 마셨다"에서 "마셨다"는 "커피를"과 문법적으로 연결되고, "카페에서"와 장소적으로 연결되며, "어제"와 시간적으로 연결됩니다.

**Multi-Head Attention**은 이런 다양한 관계를 동시에 포착하기 위해 Attention을 여러 개(보통 4~16개) 병렬로 수행합니다. 각 Head가 서로 다른 관점의 관계를 학습하고, 결과를 합쳐서 풍부한 문맥 표현을 만듭니다.

```
Head 1: 문법적 관계에 주목  ("마셨다" ↔ "커피를")
Head 2: 장소 관계에 주목    ("마셨다" ↔ "카페에서")
Head 3: 시간 관계에 주목    ("마셨다" ↔ "어제")
Head 4: 감성/톤에 주목      ("맛있는" ↔ "커피를")

→ 4개의 결과를 합쳐서 (Concat) 최종 출력
```

코드에서는 `num_heads` 파라미터로 Head 수를 지정합니다. `d_model`(벡터 차원)이 `num_heads`로 나누어 떨어져야 합니다.


---
## 4. Positional Encoding — 단어 순서 정보 추가

LSTM은 단어를 순서대로 하나씩 읽기 때문에 위치 정보를 자동으로 알 수 있습니다.
하지만 Transformer는 모든 단어를 동시에 처리하므로, 그대로 두면 "나는 너를 좋아한다"와 "너를 나는 좋아한다"를 구별할 수 없습니다.

이 문제를 해결하기 위해 **각 위치(1번째, 2번째, ...)에 고유한 패턴 벡터를 더해줍니다.**
sin과 cos 함수를 사용하여 각 위치마다 고유한 파형을 만들고, 이를 임베딩 벡터에 더합니다.

```
짝수 차원: PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
홀수 차원: PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

이렇게 하면 가까운 위치끼리는 비슷한 패턴, 먼 위치끼리는 다른 패턴을 가지게 되어 모델이 단어 순서를 인식할 수 있습니다. Positional Encoding은 학습되는 값이 아니라 수학적으로 정해진 고정값입니다.


---
## 5. Transformer Encoder의 전체 구조

Transformer Encoder는 아래 구조를 여러 번(num_layers) 반복 쌓은 것입니다.

```
입력 (정수 시퀀스)
  ↓
[Embedding]  각 단어를 d_model 차원 벡터로 변환
  ↓
[+ Positional Encoding]  위치 정보를 더함
  ↓
┌─────────────────────────────────────┐
│  [Multi-Head Self-Attention]        │ ← 단어 간 관계 계산
│  [Add & LayerNorm]                  │ ← 잔차 연결 + 정규화
│  [Feed-Forward Network]             │ ← 비선형 변환 (차원 확장 후 축소)
│  [Add & LayerNorm]                  │ ← 잔차 연결 + 정규화
└─────────────────────────────────────┘
            × num_layers 반복
  ↓
[Mean Pooling]  모든 단어 벡터의 평균 → 문장 벡터 1개
  ↓
[Linear + Sigmoid]  분류 출력
```

### 각 구성 요소의 역할

| 구성 요소 | 역할 |
|-----------|------|
| Embedding | 단어 정수를 밀집 벡터로 변환 |
| Positional Encoding | 단어 순서 정보를 벡터에 추가 |
| Multi-Head Attention | 여러 관점에서 단어 간 관계를 파악 |
| Feed-Forward Network | 각 위치에서 독립적으로 비선형 변환 수행 |
| Add & LayerNorm | 학습 안정화 (잔차 연결로 기울기 소실 방지) |
| Mean Pooling | 모든 위치의 출력을 평균내어 문장 전체를 대표하는 벡터 생성 |


---
## 6. Transformer 감성 분류 모델 — 코드 실행

아래 코드를 순서대로 실행하여 Transformer 모델을 학습시키고, 감성 분석 결과를 확인합니다.

### 6-1. 데이터 준비


In [ ]:
import torch
import torch.nn as nn

reviews = [
    ("이 영화 정말 재미있다 최고의 영화", 1),
    ("배우 연기가 훌륭하고 스토리도 좋다", 1),
    ("감동적인 영화 눈물이 났다", 1),
    ("올해 최고의 영화 강추한다", 1),
    ("재미있고 감동적이다 또 보고 싶다", 1),
    ("연출이 뛰어나고 음악도 좋다", 1),
    ("완벽한 영화 모든 것이 좋았다", 1),
    ("시간 가는 줄 모르고 봤다 재미있다", 1),
    ("이 영화 정말 지루하다 별로다", 0),
    ("스토리가 너무 진부하고 재미없다", 0),
    ("시간 낭비 돈 아깝다", 0),
    ("연기가 어색하고 내용이 별로다", 0),
    ("최악의 영화 다시는 안 본다", 0),
    ("지루하고 졸렸다 비추천한다", 0),
    ("기대했는데 실망이다 별로다", 0),
    ("돈이 아깝다 최악이다", 0),
]

all_words = [w for text, _ in reviews for w in text.split()]
vocab = sorted(set(all_words))
word2idx = {w: i+2 for i, w in enumerate(vocab)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1

MAX_LEN = 10

def encode(text):
    ids = [word2idx.get(w, 1) for w in text.split()]
    ids = ids[:MAX_LEN]
    ids = ids + [0] * (MAX_LEN - len(ids))
    return ids

X = torch.tensor([encode(t) for t, _ in reviews], dtype=torch.long)
y = torch.tensor([label for _, label in reviews], dtype=torch.float32)
VOCAB_SIZE = len(word2idx)

print(f"어휘 크기: {VOCAB_SIZE}개 | X shape: {X.shape} | y shape: {y.shape}")


### 6-2. 모델 정의


In [ ]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        mask = (x == 0)
        x = self.embedding(x)
        x = self.pos_enc(x)
        x = self.encoder(x, src_key_padding_mask=mask)
        x = x.mean(dim=1)
        x = torch.sigmoid(self.fc(x))
        return x.squeeze(1)

D_MODEL = 32
NUM_HEADS = 4
NUM_LAYERS = 2

model = TransformerClassifier(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS)
print(model)
print(f"\n총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}개")


### 6-3. 데이터 흐름 추적

문장 하나가 모델을 통과하면서 shape이 어떻게 변하는지 확인합니다.


In [ ]:
sample = "이 영화 정말 재미있다 최고의 영화"
inp = torch.tensor([encode(sample)], dtype=torch.long)

model.eval()
with torch.no_grad():
    emb = model.embedding(inp)
    pos = model.pos_enc(emb)
    enc = model.encoder(pos, src_key_padding_mask=(inp == 0))
    pooled = enc.mean(dim=1)
    prob = torch.sigmoid(model.fc(pooled))

print(f"원본 문장: {sample}")
print(f"")
print(f"[1] 인코딩+패딩  →  {inp.shape}       정수 시퀀스")
print(f"[2] Embedding    →  {emb.shape}   각 정수 → {D_MODEL}차원 벡터")
print(f"[3] + PosEnc     →  {pos.shape}   위치 정보 추가 (값만 변화)")
print(f"[4] Transformer  →  {enc.shape}   Self-Attention으로 문맥 반영")
print(f"[5] Mean Pool    →  {pooled.shape}        문장 벡터 1개")
print(f"[6] Linear+Sig   →  {prob.item():.4f}            긍정 확률")


### 6-4. 학습 및 예측


In [ ]:
# 모델 학습
model = TransformerClassifier(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 150
losses = []

for epoch in range(EPOCHS):
    model.train()
    pred = model(X)
    loss = criterion(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if (epoch + 1) % 30 == 0:
        acc = ((pred >= 0.5).float() == y).float().mean()
        print(f"  Epoch {epoch+1:3d}/{EPOCHS} | 손실: {loss.item():.4f} | 정확도: {acc.item():.2%}")

print("\n학습 완료!")


In [ ]:
# 학습 곡선 시각화
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(losses, color='blue', linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Transformer 학습 손실 곡선")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 새 문장으로 예측
def predict(text):
    model.eval()
    inp = torch.tensor([encode(text)], dtype=torch.long)
    with torch.no_grad():
        prob = model(inp).item()
    label = "긍정" if prob >= 0.5 else "부정"
    emoji = '😊' if label == '긍정' else '😞'
    print(f"  '{text}'  →  {label} {emoji} ({prob:.0%})")

print("=== 감성 분석 결과 ===")
predict("정말 재미있다 최고의 영화")
predict("지루하고 별로다 실망이다")
predict("연기가 좋고 감동적이다")
predict("시간 낭비 최악이다")


### 6-5. Attention 히트맵 시각화

학습된 모델이 실제로 어떤 단어에 주목하는지 확인합니다.


In [ ]:
import matplotlib.pyplot as plt

def show_attention(text):
    model.eval()
    inp = torch.tensor([encode(text)], dtype=torch.long)
    tokens = text.split()
    n = len(tokens)

    with torch.no_grad():
        emb = model.pos_enc(model.embedding(inp))
        first_layer = model.encoder.layers[0]
        _, attn_weights = first_layer.self_attn(
            emb, emb, emb,
            key_padding_mask=(inp == 0)
        )

    w = attn_weights[0, :n, :n].numpy()

    plt.figure(figsize=(max(4, n), max(3, n * 0.7)))
    plt.imshow(w, cmap="Blues")
    plt.xticks(range(n), tokens)
    plt.yticks(range(n), tokens)
    plt.xlabel("참조되는 단어 (Key)")
    plt.ylabel("질문하는 단어 (Query)")
    plt.title(f"Attention 히트맵: {text}")

    for i in range(n):
        for j in range(n):
            color = "white" if w[i,j] > 0.35 else "black"
            plt.text(j, i, f"{w[i,j]:.2f}",
                     ha="center", va="center", fontsize=9, color=color)

    plt.colorbar(fraction=0.046)
    plt.tight_layout()
    plt.show()

show_attention("정말 재미있다 최고의 영화")
show_attention("지루하고 별로다 실망이다")


---
## 7. Transformer에서 ChatGPT까지 — 답변이 생성되는 원리

### 7-1. Encoder vs Decoder

Transformer에는 Encoder와 Decoder 두 가지 구조가 있습니다.

| 구분 | Encoder | Decoder |
|------|---------|--------|
| Attention 방향 | **양방향** — 모든 단어를 동시에 참조 | **단방향** — 왼쪽(이전) 단어만 참조 |
| 대표 모델 | BERT | GPT, ChatGPT |
| 주요 용도 | 분류, 이해, 정보 추출 | 텍스트 생성, 대화 |

위에서 만든 감성 분류 모델은 Encoder를 사용했습니다. 문장 전체를 양방향으로 보고 "긍정/부정"이라는 하나의 결과를 출력합니다.

ChatGPT는 Decoder를 사용합니다. 왼쪽 단어들만 보면서 **다음에 올 단어를 예측**하고, 이를 반복해서 문장을 생성합니다.

### 7-2. ChatGPT가 답변하는 과정

ChatGPT의 핵심 원리는 **"다음 단어 예측을 반복"**하는 것입니다.

```
사용자 입력: "오늘 날씨 어때?"

Step 1: "오늘 날씨 어때?"              → Self-Attention → 다음 단어 = "오늘"
Step 2: "오늘 날씨 어때? 오늘"          → Self-Attention → 다음 단어 = "날씨는"
Step 3: "오늘 날씨 어때? 오늘 날씨는"    → Self-Attention → 다음 단어 = "맑아요"
Step 4: "오늘 날씨 어때? 오늘 날씨는 맑아요" → Self-Attention → 다음 단어 = "."
Step 5: "." = 종료 토큰 → 생성 중단

최종 응답: "오늘 날씨는 맑아요."
```

매 단계에서 Self-Attention으로 지금까지 나온 모든 단어의 관계를 파악하고, 가장 자연스러운 다음 단어를 선택합니다. 이것을 수백~수천 번 반복하면 긴 답변이 완성됩니다.

위에서 만든 감성 분류 모델의 마지막 출력을 "긍정/부정 확률" 대신 "전체 어휘에 대한 다음 단어 확률 분포"로 바꾸면, 바로 GPT의 기본 구조가 됩니다.

### 7-3. 왜 ChatGPT의 답변이 자연스러운가?

GPT는 인터넷의 방대한 텍스트 데이터로 **사전학습(Pre-training)**을 합니다.
수십억 개의 문장에서 "이 단어 다음에는 어떤 단어가 올 확률이 높은가?"를 학습합니다.
이 과정에서 문법, 상식, 논리 등이 Self-Attention의 가중치 속에 자연스럽게 녹아듭니다.

추가로 **RLHF(인간 피드백 강화학습)**을 통해 사람이 선호하는 답변 스타일을 학습시키면, 단순한 다음 단어 예측을 넘어 유용하고 자연스러운 대화가 가능해집니다.


---
## 8. 딥러닝 구조 이해 → RAG 및 파인튜닝 이해

Transformer의 구조를 이해하면, 향후 배울 RAG와 파인튜닝이 무엇을 하는 것인지 자연스럽게 파악할 수 있습니다.

### 8-1. 사전학습 → 파인튜닝 → RAG

```
[사전학습 (Pre-training)]
  대규모 텍스트로 "다음 단어 예측" 학습
  → 범용적인 언어 능력을 가진 기본 모델 생성
  → 예: GPT-4, Claude 등의 기반 모델
        ↓
[파인튜닝 (Fine-tuning)]
  사전학습된 모델의 가중치를 특정 도메인 데이터로 추가 학습
  → 의료, 법률, 금융 등 전문 분야에 특화된 모델로 변환
  → Transformer의 가중치(W_Q, W_K, W_V 등)가 미세 조정됨
        ↓
[RAG (Retrieval-Augmented Generation)]
  외부 문서를 검색하여 프롬프트에 함께 넣어줌
  → 모델 가중치를 바꾸지 않고도 최신/전문 정보를 활용 가능
  → Transformer의 임베딩 벡터로 문서 유사도를 검색
```

### 8-2. Transformer 구조와의 연결

| 개념 | Transformer와의 관계 |
|------|---------------------|
| **사전학습** | Transformer의 가중치(Attention, FFN 등)를 대규모 데이터로 최적화하는 과정 |
| **파인튜닝** | 사전학습된 Transformer의 가중치를 특정 데이터로 미세 조정하는 과정 |
| **임베딩** | Transformer 내부의 벡터 표현. 문서를 벡터로 변환하여 유사도를 계산할 수 있음 |
| **RAG** | 임베딩으로 관련 문서를 검색 → 검색 결과를 Transformer 입력에 추가하여 답변 생성 |

즉, Transformer의 Self-Attention 구조가 언어를 이해하는 핵심이고,
파인튜닝은 이 구조의 가중치를 조정하는 것이며,
RAG는 이 구조에 외부 지식을 주입하는 방법입니다.


---
## 전체 핵심 정리

### Transformer 핵심 개념

| 개념 | 핵심 설명 |
|------|----------|
| Self-Attention | 모든 단어 쌍의 관련도를 계산하여 문맥을 반영 |
| Q, K, V | Query(질문), Key(키), Value(값) — 같은 입력을 세 역할로 변환 |
| Multi-Head | 여러 관점에서 동시에 Attention 수행 |
| Positional Encoding | sin/cos 함수로 단어 순서 정보를 추가 |
| Encoder | 양방향 Attention → 분류/이해 (BERT) |
| Decoder | 단방향 Attention → 텍스트 생성 (GPT) |

### Self-Attention 수식

```
Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```

### ChatGPT 답변 원리

```
Self-Attention으로 문맥 파악 → 다음 단어 확률 예측 → 반복 → 답변 완성
```

### 발전 경로

```
BoW/TF-IDF → Word2Vec → RNN/LSTM → Transformer → GPT → ChatGPT
(빈도)       (의미)     (순서)      (동시+관계)   (생성)  (대화)
```
